# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjoyy/ml-flyrankai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Scoring / Ranking** — not classification, not clustering.

My lane (CTR / Engagement Opportunity Scoring) asks: *Which visible pages under-capture clicks and deserve review first?* The output is a **score per page** that ranks the entire inventory by CTR improvement opportunity. A content team then works down that ranked list in priority order.

Why not classification? A binary "has opportunity / doesn't" loses the ordering — the team needs to know *which 20 pages to start with*, not just which half of the inventory qualifies. Why not clustering? I'm not discovering groups; I'm scoring every page against a position-adjusted expectation. Scoring fits the decision: spend limited editorial time on the pages where a metadata rewrite is most likely to yield additional clicks.

In [ ]:
print('Task type: Scoring / Ranking')
print('Output: One continuous score per page, ranked high to low')
print('User of the output: Content/SEO team with limited review capacity')
print('Decision: Which pages to rewrite title/meta first')

## 2. Target or proxy

**Target:** A binary label — `low_ctr_opportunity` — defined as: a page whose observed CTR is substantially below the expected CTR for its position tier.

**Definition (the rule that creates the label):**

```
low_ctr_opportunity = 1  when  impressions_90d >= 500
                        AND  avg_position > 0  AND  avg_position <= 20
                        AND  ctr < (tier_median_ctr * 0.5)
low_ctr_opportunity = 0  otherwise
```

Where `tier_median_ctr` is the median CTR among pages in the same `position_tier` (top_3, striking, page_1, etc.) with at least 500 impressions.

**Where the label comes from:** This is a *defined proxy*, not an observed future outcome. It captures the idea of "CTR is unusually low for where this page ranks" — but it is not proof that a rewrite will fix it. A stronger label would require before/after experiment data we do not have. I am being honest: this proxy says "worth reviewing," not "will improve if changed."

In [ ]:
import pandas as pd
import numpy as np
import os

candidates = [
    os.path.join('data', 'raw', 'content_refresh_anonymized.csv'),
    os.path.join('..', '..', 'data', 'raw', 'content_refresh_anonymized.csv'),
]
data_path = next(p for p in candidates if os.path.exists(p))
df = pd.read_csv(data_path)

visible = df[df['impressions_90d'] >= 500].copy()
visible = visible[visible['avg_position'] > 0].copy()

tier_medians = visible.groupby('position_tier')['ctr'].median()
print('Median CTR by position tier (pages with >= 500 impressions):')
print(tier_medians.round(3).to_string())

visible = visible.merge(tier_medians.rename('tier_median_ctr'), on='position_tier', how='left')
visible['low_ctr_opportunity'] = (
    (visible['avg_position'] <= 20) &
    (visible['ctr'] < visible['tier_median_ctr'] * 0.5)
).astype(int)

print(f'\nLabel distribution (visible pages, pos <= 20):')
pos_le20 = visible[visible['avg_position'] <= 20]
print(pos_le20['low_ctr_opportunity'].value_counts().sort_index())
print(f'\nPositive rate: {pos_le20["low_ctr_opportunity"].mean()*100:.1f}%')

## 3. Success metric

**Primary metric: Precision@20** — of the top 20 pages the model ranks highest, how many actually have a genuine CTR opportunity (label = 1)?

**Why Precision@20?**
- The content team likely reviews 15-25 pages per week. Precision@20 directly measures how often the team's time is well-spent.
- It matches the real workflow: the team opens the ranked list, checks the top 20, and acts.
- It is robust to class imbalance (most pages are NOT opportunities) and avoids the misleading accuracy you get from a naive classifier.

**What number means "good"?**
- Baseline (random or fixed rule): ~20-30% precision (matching the base rate of opportunities among visible pages).
- Good: >= 50% precision@20 — the model is right about at least half the pages it recommends.
- Strong: >= 70% precision@20 — the team can act on most of the top 20 with confidence.

I will also track **Average Precision** across the full ranked list to measure overall ranking quality, but Precision@20 is the metric I defend.

In [ ]:
print('Primary metric: Precision@20')
print('  - Measures: of top 20 ranked pages, how many are true opportunities?')
print('  - Matches workflow: team reviews ~20 pages per sprint')
print('  - Baseline (random): ~{:.0f}% (base rate among visible pages)'.format(
    pos_le20['low_ctr_opportunity'].mean() * 100))
print('  - Good: >= 50%')
print('  - Strong: >= 70%')
print('\nSecondary metric: Average Precision (full ranked list quality)')

## 4. The unit of analysis, as a real dataframe

**One row = one content page.**

The starter dataset already has this grain: 30,000 rows, each representing one pseudonymized content item with its 90-day search and engagement metrics. The dataframe below shows the features that matter for this lane — position, CTR, impressions, engagement rate — plus the label I defined above.

The key columns for CTR opportunity scoring:
- `avg_position` — where the page ranks (lower is better)
- `ctr` — observed click-through rate
- `impressions_90d` — volume of search visibility
- `position_tier` — which position bucket the page falls in
- `content_type` — keyword article, feedly article, comparison article
- `main_intent` — informational, transactional, commercial, navigational
- `low_ctr_opportunity` — my defined label (1 = opportunity, 0 = not)

In [ ]:
columns_to_show = [
    'content_id', 'client_id', 'avg_position', 'position_tier', 'ctr',
    'impressions_90d', 'clicks_90d', 'content_type', 'main_intent',
    'engagement_rate', 'sessions_90d', 'low_ctr_opportunity'
]
show_df = visible[columns_to_show].copy()

print(f'Unit of analysis: one row = one content page')
print(f'Dataframe shape (visible pages with position data, pos <= 20): {show_df[show_df["avg_position"] <= 20].shape}')
print(f'\nFirst 10 rows of the lane slice:')
show_df[show_df['avg_position'] <= 20].head(10)

## 5. Why ML beats a fixed rule here

A fixed rule like "CTR < 1% and position <= 20" sounds simple, but it has three problems the data exposes:

**Problem 1: CTR expectations differ by position tier.** A page at position 3 with 0.5% CTR is in a very different situation than a page at position 18 with 0.5% CTR. The top_3 tier naturally gets higher CTR — so 0.5% there is a bigger red flag than 0.5% at position 18. A fixed threshold ignores this.

**Problem 2: Multiple signals interact.** Position, impressions, content type, intent, and engagement rate all affect whether a low CTR is a real opportunity or just a niche query. A keyword article at position 5 with 10,000 impressions and 0.1% CTR is a very different case than a feedly article at position 12 with 600 impressions and 0.3% CTR. An if-statement can capture one threshold; ML captures the joint pattern.

**Problem 3: The base rate shifts.** Not every visible page is worth reviewing. If the rule flags too many pages, the team wastes time. If it flags too few, opportunities are missed. A learned model can tune the decision boundary to match the team's actual capacity.

The code below shows the problem with a single threshold: CTR distribution varies dramatically across position tiers, so one cutoff creates both false positives and false negatives.

In [ ]:
print('=== WHY A SINGLE CTR THRESHOLD FAILS ===')
print('\nCTR distribution varies by position tier:')
tier_stats = visible[visible['avg_position'] <= 20].groupby('position_tier').agg(
    pages=('ctr', 'count'),
    median_ctr=('ctr', 'median'),
    mean_ctr=('ctr', 'mean'),
    p25_ctr=('ctr', lambda x: x.quantile(0.25)),
    p75_ctr=('ctr', lambda x: x.quantile(0.75)),
).round(3)
print(tier_stats.to_string())

print('\n--- Single threshold problem ---')
threshold = 1.0
flagged = visible[(visible['avg_position'] <= 20) & (visible['ctr'] < threshold)]
print(f'Rule: CTR < {threshold}% AND position <= 20')
print(f'Pages flagged: {len(flagged)} ({len(flagged)/len(pos_le20)*100:.1f}% of eligible pages)')
print(f'Of those, actual opportunities: {flagged["low_ctr_opportunity"].sum()} ({flagged["low_ctr_opportunity"].mean()*100:.1f}% precision)')

print('\n--- Position-tier-aware rule ---')
for tier in ['top_3', 'striking', 'page_1']:
    tier_pages = pos_le20[pos_le20['position_tier'] == tier]
    cutoff = tier_medians[tier] * 0.5
    tier_flagged = tier_pages[tier_pages['ctr'] < cutoff]
    if len(tier_flagged) > 0:
        prec = tier_flagged['low_ctr_opportunity'].mean()*100
    else:
        prec = 0
    print(f'  {tier}: median CTR = {tier_medians[tier]:.3f}%, cutoff = {cutoff:.3f}%, '
          f'flagged = {len(tier_flagged)}, precision = {prec:.1f}%')

print('\nConclusion: the right cutoff changes per tier.')
print('A learned model captures this automatically — and adds content_type, intent, engagement, and volume.')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.